# AICE 에러 해결 가이드 (Troubleshooting)

> 시험 중 흔하게 마주치는 에러 메시지와 그 원인, 해결 방법을 정리했습니다.

### 🚨 ValueError: could not convert string to float
- **원인:** 모델 학습(`fit`)이나 스케일링을 할 때 문자열 컬럼이 포함되어 발생합니다.
- **해결:** 범주형(문자열) 컬럼을 삭제하거나 `get_dummies`로 인코딩하세요.
```python
# 에러 유발 컬럼 확인
print(X_train.dtypes)
# 인코딩 처리
df = pd.get_dummies(df, drop_first=True)
```

### 🚨 ValueError: Found array with dim 3. Estimator expected <= 2.
- **원인:** Scikit-learn 모델에 3차원 이상의 배열을 넣었을 때 발생합니다. (주로 이미지 데이터)
- **해결:** `reshape()`를 사용하여 2차원으로 펼쳐주세요.
```python
X_train_flat = X_train.reshape(X_train.shape[0], -1)
```

### 🚨 ValueError: Input contains NaN, infinity or a value too large for dtype('float64').
- **원인:** 데이터에 결측치(NaN)가 아직 남아있는데 모델 학습을 시도했을 때 발생합니다.
- **해결:** 결측치를 완전히 제거(`dropna`)하거나 적절한 값으로 채우세요(`fillna`).
```python
print(df.isnull().sum()) # 결측치가 남은 컬럼 확인
df = df.fillna(df.mean()) # 평균값으로 채우기
```

### 🚨 AttributeError: 'Series' object has no attribute 'flatten'
- **원인:** pandas Series 객체에는 numpy 배열용 메서드인 flatten이 없습니다.
- **해결:** Series를 numpy 배열로 바꾼 뒤 flatten을 적용하세요.
```python
# 잘못된 방법
# y_test.flatten()
# 올바른 방법
y_test.values.flatten()
```

### 🚨 Shape mismatch (스케일링 관련 에러)
- **원인:** Train과 Test의 컬럼 수가 다를 때 (예: Test 데이터에 특정 범주가 없어서 get_dummies 결과 컬럼 수가 달라짐) 발생합니다.
- **해결:** get_dummies를 할 때 Train과 Test를 합쳐서(concat) 한 번에 인코딩한 후 다시 분리하세요.
```python
# 데이터를 분할하기 전에 인코딩을 먼저 수행하는 것이 안전합니다.
df = pd.get_dummies(df, drop_first=True)
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```
